# Lab 1: Linear Regression -- From Scratch to Practice

In this notebook, you will:
1. Implement OLS linear regression from scratch using NumPy
2. Verify your implementation against scikit-learn
3. Apply linear regression to the Ames Housing dataset
4. Perform residual diagnostics and assumption checking
5. Interpret coefficients in a real-world context

**Prerequisites:** Modules 1 and 2 (Linear Regression concepts and mathematics)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Part 1: Implementing OLS from Scratch

Recall from Module 2 that the OLS solution is:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

We will implement this directly.

In [ ]:
class LinearRegressionFromScratch:
    """OLS Linear Regression implemented using the Normal Equation."""
    
    def __init__(self):
        self.beta = None
        self.residuals = None
    
    def fit(self, X, y):
        # Add intercept column (column of ones)
        n = X.shape[0]
        X_aug = np.column_stack([np.ones(n), X])
        
        # Normal equation: beta = (X^T X)^{-1} X^T y
        XtX = X_aug.T @ X_aug
        Xty = X_aug.T @ y
        self.beta = np.linalg.solve(XtX, Xty)  # More stable than explicit inverse
        
        # Compute residuals
        y_pred = X_aug @ self.beta
        self.residuals = y - y_pred
        return self
    
    def predict(self, X):
        n = X.shape[0]
        X_aug = np.column_stack([np.ones(n), X])
        return X_aug @ self.beta
    
    def r_squared(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - ss_res / ss_tot
    
    def mse(self, X, y):
        y_pred = self.predict(X)
        return np.mean((y - y_pred) ** 2)
    
    @property
    def intercept(self):
        return self.beta[0]
    
    @property
    def coefficients(self):
        return self.beta[1:]

### Test on synthetic data

In [ ]:
# Generate synthetic data: y = 3 + 2*x1 - 1.5*x2 + noise
n_samples = 200
X_synth = np.random.randn(n_samples, 2)
y_synth = 3 + 2 * X_synth[:, 0] - 1.5 * X_synth[:, 1] + np.random.randn(n_samples) * 0.5

# Fit our model
model = LinearRegressionFromScratch()
model.fit(X_synth, y_synth)

print("True parameters: intercept=3.0, beta1=2.0, beta2=-1.5")
print(f"Estimated:       intercept={model.intercept:.4f}, beta1={model.coefficients[0]:.4f}, beta2={model.coefficients[1]:.4f}")
print(f"R-squared: {model.r_squared(X_synth, y_synth):.4f}")
print(f"MSE: {model.mse(X_synth, y_synth):.4f}")

### Verify against scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression()
sk_model.fit(X_synth, y_synth)

print("Comparison (our implementation vs sklearn):")
print(f"  Intercept:  {model.intercept:.6f} vs {sk_model.intercept_:.6f}")
print(f"  Coef 1:     {model.coefficients[0]:.6f} vs {sk_model.coef_[0]:.6f}")
print(f"  Coef 2:     {model.coefficients[1]:.6f} vs {sk_model.coef_[1]:.6f}")
print(f"  Max difference: {np.max(np.abs(model.beta - np.r_[sk_model.intercept_, sk_model.coef_])):.2e}")

## Part 2: Gradient Descent Implementation

Now implement the iterative approach from Module 2:

$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} - \alpha \cdot \frac{1}{n}\mathbf{X}^T(\mathbf{X}\boldsymbol{\beta}^{(t)} - \mathbf{y})$$

In [ ]:
class LinearRegressionGD:
    """Linear Regression via Gradient Descent."""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, tol=1e-8):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.tol = tol
        self.beta = None
        self.loss_history = []
    
    def fit(self, X, y):
        n, p = X.shape
        X_aug = np.column_stack([np.ones(n), X])
        
        # Initialize parameters at zero
        self.beta = np.zeros(p + 1)
        self.loss_history = []
        
        for i in range(self.n_iter):
            # Predictions
            y_pred = X_aug @ self.beta
            
            # Gradient: (1/n) * X^T * (X*beta - y)
            gradient = (1 / n) * X_aug.T @ (y_pred - y)
            
            # Update
            self.beta -= self.lr * gradient
            
            # Track loss
            loss = np.mean((y_pred - y) ** 2) / 2
            self.loss_history.append(loss)
            
            # Check convergence
            if np.linalg.norm(gradient) < self.tol:
                print(f"Converged at iteration {i}")
                break
        
        return self
    
    def predict(self, X):
        n = X.shape[0]
        X_aug = np.column_stack([np.ones(n), X])
        return X_aug @ self.beta

In [ ]:
# Standardize features for gradient descent (important for convergence)
X_std = (X_synth - X_synth.mean(axis=0)) / X_synth.std(axis=0)
y_std = (y_synth - y_synth.mean()) / y_synth.std()

gd_model = LinearRegressionGD(learning_rate=0.1, n_iterations=500)
gd_model.fit(X_std, y_std)

# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(gd_model.loss_history)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss (MSE/2)')
axes[0].set_title('Gradient Descent Convergence')

axes[1].semilogy(gd_model.loss_history)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Convergence (log scale)')
plt.tight_layout()
plt.show()

### Learning Rate Experiment

Observe the effect of different learning rates on convergence behavior.

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.5]

plt.figure(figsize=(10, 5))
for lr in learning_rates:
    m = LinearRegressionGD(learning_rate=lr, n_iterations=100)
    m.fit(X_std, y_std)
    label = f'lr={lr}'
    if m.loss_history[-1] > 1e3:
        label += ' (diverges)'
    plt.plot(m.loss_history[:50], label=label)

plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Effect of Learning Rate on Convergence')
plt.legend()
plt.ylim(0, 2)
plt.show()

## Part 3: Real-World Application -- Ames Housing Dataset

We now apply linear regression to predict house sale prices using the Ames Housing dataset.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load Ames Housing data
# If fetch_openml is unavailable, generate synthetic housing data
try:
    housing = fetch_openml(name='house_prices', version=1, as_frame=True, parser='auto')
    df = housing.frame
except:
    # Synthetic fallback
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'GrLivArea': np.random.uniform(500, 4000, n),
        'BedroomAbvGr': np.random.choice([2, 3, 4, 5], n),
        'OverallQual': np.random.choice(range(1, 11), n),
        'YearBuilt': np.random.randint(1950, 2010, n),
        'GarageCars': np.random.choice([0, 1, 2, 3], n),
    })
    df['SalePrice'] = (
        50 * df['GrLivArea'] + 
        15000 * df['BedroomAbvGr'] + 
        22000 * df['OverallQual'] +
        500 * (df['YearBuilt'] - 1950) +
        18000 * df['GarageCars'] +
        np.random.randn(n) * 25000
    )

# Select features
features = ['GrLivArea', 'BedroomAbvGr', 'OverallQual', 'YearBuilt', 'GarageCars']
available_features = [f for f in features if f in df.columns]
X_house = df[available_features].values.astype(float)
y_house = df['SalePrice'].values.astype(float)

# Remove any NaN rows
mask = ~np.isnan(X_house).any(axis=1) & ~np.isnan(y_house)
X_house = X_house[mask]
y_house = y_house[mask]

print(f"Dataset: {X_house.shape[0]} samples, {X_house.shape[1]} features")
print(f"Features: {available_features}")
print(f"Price range: ${y_house.min():,.0f} to ${y_house.max():,.0f}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

# Fit our OLS model
house_model = LinearRegressionFromScratch()
house_model.fit(X_train, y_train)

# Results
r2_train = house_model.r_squared(X_train, y_train)
r2_test = house_model.r_squared(X_test, y_test)
rmse_test = np.sqrt(house_model.mse(X_test, y_test))

print(f"R-squared (train): {r2_train:.4f}")
print(f"R-squared (test):  {r2_test:.4f}")
print(f"RMSE (test):       ${rmse_test:,.0f}")
print(f"\nCoefficient Interpretation:")
print(f"  Intercept: ${house_model.intercept:,.0f}")
for feat, coef in zip(available_features, house_model.coefficients):
    print(f"  {feat}: ${coef:,.2f} per unit increase")

## Part 4: Residual Diagnostics

Check whether the assumptions from Module 1 hold for our housing model.

In [ ]:
y_pred_train = house_model.predict(X_train)
residuals = y_train - y_pred_train
standardized_residuals = (residuals - residuals.mean()) / residuals.std()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(y_pred_train, residuals, alpha=0.4, s=15)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted (check linearity + homoscedasticity)')

# 2. Q-Q Plot
stats.probplot(standardized_residuals, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('Normal Q-Q Plot (check normality)')

# 3. Scale-Location
axes[1, 0].scatter(y_pred_train, np.sqrt(np.abs(standardized_residuals)), alpha=0.4, s=15)
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('sqrt(|Standardized Residuals|)')
axes[1, 0].set_title('Scale-Location (check homoscedasticity)')

# 4. Residual histogram
axes[1, 1].hist(standardized_residuals, bins=30, density=True, alpha=0.7)
x_norm = np.linspace(-4, 4, 100)
axes[1, 1].plot(x_norm, stats.norm.pdf(x_norm), 'r-', linewidth=2)
axes[1, 1].set_xlabel('Standardized Residuals')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Residual Distribution vs Normal')

plt.tight_layout()
plt.show()

## Part 5: Exercises

1. **Feature engineering:** Add polynomial features (e.g., `GrLivArea^2`) and see if R-squared improves. Does the residual pattern change?

2. **Multicollinearity:** Compute the correlation matrix of features. Which pairs are most correlated? What happens to coefficient stability if you add a feature that is nearly a linear combination of existing features?

3. **Outlier sensitivity:** Artificially add 5 extreme outliers to the training set. How much do the coefficients change? Compare with the original model.

4. **Gradient descent convergence:** Without standardization, gradient descent converges much more slowly (or not at all). Verify this by running `LinearRegressionGD` on the raw (unstandardized) housing features. Why does this happen? (Hint: think about the condition number of X^T X.)

In [ ]:
# Exercise workspace -- try the exercises above here

# Example: correlation matrix
corr_matrix = np.corrcoef(X_train.T)
print("Feature correlation matrix:")
print(pd.DataFrame(corr_matrix, columns=available_features, index=available_features).round(3))